Задание 1 (синтаксис)

Напишите функцию `check_car_id(car_id: str) → str`.

Формат номера: 1 буква + 3 цифры + 2 буквы + 2–3 цифры региона.
Допустимые буквы: А, В, Е, К, М, Н, О, Р, С, Т, У, Х (кириллица).

Если номер валиден: «Номер А222ВС валиден. Регион: 96.»
Если невалиден: «Номер не валиден.»

Правило нормализации: латинские A, B, E, K, M, H, O, P, C, T, Y, X
заменяются на визуально похожие кириллические буквы. Строчные буквы переводятся
в верхний регистр. Пробелы не удаляются. Цифры — только 0–9.
Проверяется синтаксис номера, а не существование кода региона.


In [1]:
import re


def check_car_id(car_id: str) -> str:
    latin_to_cyrillic = str.maketrans("ABEKMHOPCTYX", "АВЕКМНОРСТУХ")
    normalized_car_id = car_id.upper().translate(latin_to_cyrillic)
    match = re.fullmatch(
        r"([АВЕКМНОРСТУХ][0-9]{3}[АВЕКМНОРСТУХ]{2})([0-9]{2,3})",
        normalized_car_id,
    )

    if match is None:
        return "Номер не валиден."

    number, region = match.groups()
    return f"Номер {number} валиден. Регион: {region}."


In [2]:
car_ids = [
    "А222ВС96", "А123ВЕ777", "АБ22ВВ193", "A222BC96",
    "А222ВС9", "А222ВС9699", "Я222ВС96", "",
]

for car_id in car_ids:
    print(f"{car_id!r}: {check_car_id(car_id)}")


'А222ВС96': Номер А222ВС валиден. Регион: 96.
'А123ВЕ777': Номер А123ВЕ валиден. Регион: 777.
'АБ22ВВ193': Номер не валиден.
'A222BC96': Номер А222ВС валиден. Регион: 96.
'А222ВС9': Номер не валиден.
'А222ВС9699': Номер не валиден.
'Я222ВС96': Номер не валиден.
'': Номер не валиден.


Самопроверка задания 1

Проверяем примеры из условия, смешанную раскладку, строчные буквы,
пробелы, лишние символы и цифры другого алфавита.


In [3]:
assert check_car_id("А222ВС96") == "Номер А222ВС валиден. Регион: 96."
assert check_car_id("А123ВЕ777") == "Номер А123ВЕ валиден. Регион: 777."
assert check_car_id("A222BC96") == "Номер А222ВС валиден. Регион: 96."
assert check_car_id("а222bс96") == "Номер А222ВС валиден. Регион: 96."

for car_id in [
    "АБ22ВВ193", "А222ВС9", "А222ВС9699", "Я222ВС96", "",
    " А222ВС96", "А222ВС96\n", "А٢٢٢ВС96", "А222ВС96extra",
]:
    assert check_car_id(car_id) == "Номер не валиден.", car_id

print("Все проверки задания 1 пройдены.")


Все проверки задания 1 пройдены.


Задание 2 (повышенная сложность)

Реализуйте `parse_logs(text: str) → list[dict]` для многострочного лога.

1. Регулярным выражением извлеките строки формата `YYYY-MM-DD HH:MM:SS LEVEL ...`.
2. Из хвоста строки достаньте `user`, `action`, `ip` и необязательное `card`.
3. Через regex проверьте email, IPv4 и карту формата `XXXX-XXXX-XXXX-XXXX`,
   где X — цифра. В результате храните только `****-****-****-1111`.
4. Невалидные поля сохраняйте с ошибками в `errors`, не прерывая разбор.
5. Бонус: удалите последовательные повторы слов в сообщении через `re`.

Принятые правила:

- Начальные пробелы допускаются; строки с другим форматом даты пропускаются.
  Проверяется формат времени, календарная корректность даты отдельно не проверяется.
- LEVEL — слово из заглавных латинских букв.
- Поля задаются как `ключ=значение` без пробелов внутри значения, порядок произвольный.
  При повторении ключа используется последнее значение.
- Отсутствующие или пустые `user`, `action`, `ip` дают `None` и `missing_email`,
  `missing_action`, `missing_ip` соответственно.
- Email: латинские буквы, цифры и символы `_ % + -` в локальной части;
  точки разделяют непустые части. Домен содержит хотя бы одну точку,
  его части не начинаются и не заканчиваются дефисом, зона — минимум две буквы.
- IPv4: четыре числа 0–255, без ведущих нулей, кроме самого `0`.
- Отсутствующая карта даёт `None` без ошибки. Неверный формат, включая карту
  без дефисов, даёт `None` и `invalid_card`. Полный номер карты не попадает в результат.
- Текст после `message=` считается сообщением и занимает остаток строки.
  При наличии сообщения добавляется поле `message`. Повторы удаляются без учёта регистра,
  написание первого слова сохраняется.


In [4]:
import re


def parse_logs(text: str) -> list[dict]:
    line_pattern = re.compile(
        r"^[ \t]*(?P<ts>[0-9]{4}-[0-9]{2}-[0-9]{2} "
        r"[0-9]{2}:[0-9]{2}:[0-9]{2})[ \t]+"
        r"(?P<level>[A-Z]+)(?:[ \t]+(?P<tail>[^\r\n]*))?\r?$",
        re.MULTILINE,
    )
    field_pattern = re.compile(r"(?:^|\s)(user|action|ip|card)=([^\s]*)")
    email_pattern = re.compile(
        r"[A-Za-z0-9_%+-]+(?:\.[A-Za-z0-9_%+-]+)*@"
        r"(?:[A-Za-z0-9](?:[A-Za-z0-9-]*[A-Za-z0-9])?\.)+"
        r"[A-Za-z]{2,}"
    )
    octet = r"(?:25[0-5]|2[0-4][0-9]|1[0-9]{2}|[1-9]?[0-9])"
    ip_pattern = re.compile(rf"{octet}(?:\.{octet}){{3}}")
    card_pattern = re.compile(r"[0-9]{4}(?:-[0-9]{4}){3}")
    logs = []

    for match in line_pattern.finditer(text):
        tail = match.group("tail") or ""
        message_match = re.search(r"(?:^|\s)message=", tail)
        fields_text = tail[:message_match.start()] if message_match else tail
        fields = dict(field_pattern.findall(fields_text))
        record = {
            "ts": match.group("ts"),
            "level": match.group("level"),
            "user": fields.get("user") or None,
            "action": fields.get("action") or None,
            "ip": fields.get("ip") or None,
            "card": None,
            "errors": [],
        }

        if record["user"] is None:
            record["errors"].append("missing_email")
        elif email_pattern.fullmatch(record["user"]) is None:
            record["errors"].append("invalid_email")

        if record["action"] is None:
            record["errors"].append("missing_action")

        if record["ip"] is None:
            record["errors"].append("missing_ip")
        elif ip_pattern.fullmatch(record["ip"]) is None:
            record["errors"].append("invalid_ip")

        if "card" in fields:
            if card_pattern.fullmatch(fields["card"]):
                record["card"] = "****-****-****-" + fields["card"][-4:]
            else:
                record["errors"].append("invalid_card")

        if message_match:
            message = tail[message_match.end():].strip()
            record["message"] = re.sub(
                r"\b(\w+)(?:\s+\1\b)+", r"\1", message, flags=re.IGNORECASE
            )

        logs.append(record)

    return logs


In [5]:
from pprint import pprint

text = """2024-03-15 14:22:01 INFO user=ivan@mail.ru action=login ip=192.168.0.12
 2024-03-15 14:22:05 WARN user=ivan@mail.ru action=retry ip=192.168.0.12
 2024-03-15 14:23:11 ERROR user=bad..email@ action=pay ip=999.1.1.1 card=4111-1111-1111-1111
 Mar 15 2024 14:24:00 INFO user=olga@company.com action=logout ip=10.0.0.5"""

pprint(parse_logs(text), sort_dicts=False)


[{'ts': '2024-03-15 14:22:01',
  'level': 'INFO',
  'user': 'ivan@mail.ru',
  'action': 'login',
  'ip': '192.168.0.12',
  'card': None,
  'errors': []},
 {'ts': '2024-03-15 14:22:05',
  'level': 'WARN',
  'user': 'ivan@mail.ru',
  'action': 'retry',
  'ip': '192.168.0.12',
  'card': None,
  'errors': []},
 {'ts': '2024-03-15 14:23:11',
  'level': 'ERROR',
  'user': 'bad..email@',
  'action': 'pay',
  'ip': '999.1.1.1',
  'card': '****-****-****-1111',
  'errors': ['invalid_email', 'invalid_ip']}]


Самопроверка задания 2

Проверяем результат примера, пустой ввод, отсутствующие поля, ошибки email,
границы IPv4, формат и маскирование карты, продолжение разбора после ошибки
и удаление повторов слов.


In [6]:
expected_logs = [
    {
        "ts": "2024-03-15 14:22:01", "level": "INFO",
        "user": "ivan@mail.ru", "action": "login", "ip": "192.168.0.12",
        "card": None, "errors": [],
    },
    {
        "ts": "2024-03-15 14:22:05", "level": "WARN",
        "user": "ivan@mail.ru", "action": "retry", "ip": "192.168.0.12",
        "card": None, "errors": [],
    },
    {
        "ts": "2024-03-15 14:23:11", "level": "ERROR",
        "user": "bad..email@", "action": "pay", "ip": "999.1.1.1",
        "card": "****-****-****-1111", "errors": ["invalid_email", "invalid_ip"],
    },
]
assert parse_logs(text) == expected_logs
assert parse_logs("") == []
assert parse_logs("   \n\t") == []
assert parse_logs("Mar 15 2024 14:24:00 INFO user=olga@company.com") == []

prefix = "2024-03-15 14:22:01 INFO "
missing = parse_logs(prefix)[0]
assert missing["errors"] == ["missing_email", "missing_action", "missing_ip"]
assert missing["user"] is missing["action"] is missing["ip"] is None
assert parse_logs(prefix + "user= action= ip=")[0]["errors"] == missing["errors"]

for ip in ["256.0.0.1", "999.1.1.1", "1.2.3", "1.2.3.4.5", "01.2.3.4"]:
    record = parse_logs(prefix + f"user=a@mail.ru action=login ip={ip}")[0]
    assert record["errors"] == ["invalid_ip"], ip

for ip in ["0.0.0.0", "255.255.255.255", "10.0.0.5"]:
    assert parse_logs(prefix + f"user=a@mail.ru action=login ip={ip}")[0]["errors"] == []

for email in ["bad..email@", "a..b@mail.ru", "a@-mail.ru", "a@mail", "a@mail..ru"]:
    record = parse_logs(prefix + f"user={email} action=login ip=1.2.3.4")[0]
    assert record["errors"] == ["invalid_email"], email

valid_tail = "user=olga+test@company.com action=pay ip=10.0.0.5"
for card in ["4111111111111111", "abcd-1111-1111-1111", ""]:
    record = parse_logs(prefix + valid_tail + f" card={card}")[0]
    assert record["card"] is None
    assert record["errors"] == ["invalid_card"]

record = parse_logs(prefix + valid_tail + " card=4111-1111-1111-1111")[0]
assert record["card"] == "****-****-****-1111"
assert "4111-1111-1111-1111" not in str(record)
assert record["errors"] == []

bonus = parse_logs(prefix + valid_tail + " message=error error error timeout")[0]
assert bonus["message"] == "error timeout"
assert parse_logs(prefix + valid_tail + " message=Error error timeout")[0]["message"] == "Error timeout"
assert parse_logs(prefix + valid_tail + " message=ip=999.1.1.1")[0]["ip"] == "10.0.0.5"
assert len(parse_logs(prefix + "ip=256.0.0.1\n" + prefix + valid_tail)) == 2

print("Все проверки задания 2 пройдены.")
pprint(bonus, sort_dicts=False)


Все проверки задания 2 пройдены.
{'ts': '2024-03-15 14:22:01',
 'level': 'INFO',
 'user': 'olga+test@company.com',
 'action': 'pay',
 'ip': '10.0.0.5',
 'card': None,
 'errors': [],
 'message': 'error timeout'}
